> **2026-05-11 — base model switch**: this pilot now runs on `Qwen/Qwen2.5-1.5B-Instruct` (1.5B) without the W4/W5 KTO/S-DPO chain. Wallclock drops ~2× vs the 3B run; the B1 baseline for Δ R_turn is the untrained instruct model itself.

> **🚀 SMOOTH RE-RUN MODE**: this notebook is configured for fast pilot iteration with vLLM + canonical GRPO (G=8). Cells `# 5)` install + cells `# 6)` judge + cells `# 7)` parquet build all auto-skip if their outputs already exist on Drive/Hub from a prior run. **First-time run**: ~30 min retrieval + ~30 min judge + ~25 min GRPO ≈ 90 min total. **Re-run** (data cached): ~25 min GRPO only. Revert `max_steps=500` and `num_generations=8` in cell `# 9)` for a longer/canonical run.

# 31p — W6 PILOT (Option B refactor): distilled judge + short GRPO

**Pilot purpose**: validate direction before committing 10+ A100-hr to a full W6 run. Per the user's directive: "let's run for less time so we see we are in the correct direction, not wasting precious time and money."

## What changed vs notebook 32 (the full W6)

This pilot ships the **Option B refactor** of Component B's reward signal:

1. **Trains the distilled judge first** (cell 6, ~30 min on A100 — auto-skipped on re-run if today's repo exists on Hub). Uses `data/reward_calibration_anchors.parquet` (210k labeled rows) → fine-tunes `cross-encoder/ms-marco-MiniLM-L-6-v2` on Gemini-aligned anchors.
2. **Reweights the reward** (already landed in `scripts/reward_fns.py`): R_retr 0.70→0.40, R_judge 0.10→0.30, R_format 0.05→0.10, R_user_prof 0.00→0.05. Useful gradient mass goes from ~0.20 to ~0.60. R_judge is now backed by a real model, not a stub.
3. **Adds intra-rollout diversity bonus** (`group_responses` kwarg in `compose_r_turn`, +0.05 cap): wires `compose_r_session.lex_div` into per-prompt training signal.
4. **Smaller scale**: 1500 sessions (10% of full), 500 GRPO steps (vs 12k). ~25 min on A100 with vLLM.

## Compute budget (Colab Pro: ~100 units/month)

| Step | Wallclock (first run) | Wallclock (cached re-run) | Compute units |
|---|---|---|---|
| Distilled judge training | ~30 min A100 | ~5 sec (Hub-skip) | ~6 / ~0 |
| Retrieval pre-compute (1500 sessions, ~5,295 turns) | ~30 min A100 | ~5 sec (parquet on Drive) | ~6 / ~0 |
| GRPO pilot (500 steps × G=8, vLLM) | ~25 min A100 | ~25 min A100 | ~5 / ~5 |
| Format + reward delta eval (50 rows × 2 models) | ~10 min A100 | ~10 min A100 | ~2 / ~2 |
| **Total** | **~95 min** | **~35 min** | **~19 / ~7** |

## Pilot gate (decide whether to run the full W6)

PASS if both:
- format compliance ≥ 90% (slightly relaxed from full-run 95% since pilot has fewer steps)
- Δ R_turn vs B1 (W4 merged) ≥ +0.015 on the 50-row eval

PASS → schedule full W6. FAIL → iterate (more steps, different LR, or fall back to W4-only).

In [ ]:
# 1) GPU check.
!nvidia-smi | head -10

In [ ]:
# 2) Clone fresh-model branch.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026
!git log -1 --pretty=format:'commit:  %h%nsubject: %s'

In [ ]:
# 2b) Drive + caches.
import os, shutil
from google.colab import drive
try: drive.mount('/content/drive')
except Exception:
    try: drive.flush_and_unmount()
    except Exception: pass
    drive.mount('/content/drive', force_remount=True)

DRIVE_BASE = '/content/drive/MyDrive/recsys2026-cache'
for d in [f'{DRIVE_BASE}/hf_datasets', f'{DRIVE_BASE}/experiments_cache',
          f'{DRIVE_BASE}/judge_runs', f'{DRIVE_BASE}/grpo_pilot_runs',
          f'{DRIVE_BASE}/recsys2026_data', f'{DRIVE_BASE}/recsys2026_data/trl']:
    os.makedirs(d, exist_ok=True)

os.environ['HF_DATASETS_CACHE'] = f'{DRIVE_BASE}/hf_datasets'
%env HF_DATASETS_CACHE={DRIVE_BASE}/hf_datasets

EXPECTED_CACHE = '/content/recsys2026/music-crs-baselines/experiments/cache'
os.makedirs(os.path.dirname(EXPECTED_CACHE), exist_ok=True)
if os.path.exists(EXPECTED_CACHE) and not os.path.islink(EXPECTED_CACHE):
    shutil.rmtree(EXPECTED_CACHE)
if not os.path.islink(EXPECTED_CACHE):
    os.symlink(f'{DRIVE_BASE}/experiments_cache', EXPECTED_CACHE)

# Persist all data/ outputs (anchor parquet, retrieval cache, GRPO parquet,
# judge JSONLs) on Drive so re-runs in fresh Colab sessions skip the heavy
# cell-7 retrieval pre-compute. Same pattern as experiments_cache above.
DATA_LOCAL = '/content/recsys2026/data'
DATA_DRIVE = f'{DRIVE_BASE}/recsys2026_data'
if os.path.exists(DATA_LOCAL) and not os.path.islink(DATA_LOCAL):
    # Migrate any pre-existing files from a previous (non-symlink) run.
    for fname in os.listdir(DATA_LOCAL):
        src = os.path.join(DATA_LOCAL, fname)
        dst = os.path.join(DATA_DRIVE, fname)
        if not os.path.exists(dst):
            shutil.move(src, dst)
    shutil.rmtree(DATA_LOCAL, ignore_errors=True)
if not os.path.islink(DATA_LOCAL):
    os.symlink(DATA_DRIVE, DATA_LOCAL)
print(f'✓ data/ → {DATA_DRIVE} (symlink)')

In [ ]:
# 3) HF auth — required for push_to_hub.
#
# Setup: Colab → 🔑 Secrets pane → add `HF_TOKEN` with WRITE scope.
# Get the token at https://huggingface.co/settings/tokens.
#
# Fail-fast: aborts immediately if the secret is missing or the token is
# invalid — better than failing 3 hours into training when push_to_hub fires.
import os, sys
from google.colab import userdata
from huggingface_hub import whoami

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as e:
    raise SystemExit(
        f"\u274c HF_TOKEN secret not found in Colab ({e!r}).\n"
        f"   1) Open the \U0001f511 Secrets pane in the left sidebar.\n"
        f"   2) Add a secret named exactly `HF_TOKEN` (case-sensitive).\n"
        f"   3) Toggle 'Notebook access' ON for this notebook."
    )

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

try:
    user = whoami(token=HF_TOKEN)
    HF_USERNAME = user["name"]
    os.environ["HF_USERNAME"] = HF_USERNAME
    print(f"\u2713 HF auth ok \u2014 logged in as {HF_USERNAME}")
except Exception as e:
    raise SystemExit(
        f"\u274c HF auth failed: {e!r}\n"
        f"   Token may lack WRITE scope. Regenerate at https://huggingface.co/settings/tokens"
    )

In [ ]:
# 4) Resolve starting base — prefer W4 KTO-merged Qwen-1.5B if available.
#
# Looks for a gate_result.json from a prior W4 KTO run (notebook 30_responder_kto)
# under DRIVE_BASE/kto_runs/. If found, uses its merged_hub_model. Otherwise
# falls back to the raw Qwen-2.5-1.5B-Instruct (the "no-warmup" path used in
# v2-grounded and v3.1, which hit the format ceiling at 82-84%).
#
# To force the no-KTO path: delete all gate_result.json files under
# kto_runs/, or hard-code STARTING_MERGED below.

import json
from pathlib import Path

def latest_gate(runs_dir):
    p = Path(runs_dir)
    if not p.exists():
        return None
    cands = list(p.rglob('gate_result.json'))
    if not cands:
        return None
    latest = max(cands, key=lambda x: x.stat().st_mtime)
    with latest.open() as f:
        return json.load(f)

W4 = latest_gate(f'{DRIVE_BASE}/kto_runs')

if W4 and W4.get('merged_hub_model'):
    STARTING_MERGED = W4['merged_hub_model']
    B1_REPO = STARTING_MERGED       # same model — measures pure GRPO gain over KTO
    PRIOR_STAGE = f'W4-KTO ({W4.get("run_name", "?")})'
    print(f'✓ using W4 KTO-merged base: {STARTING_MERGED}')
    print(f'  format compliance at W4 gate: {W4.get("format_compliance_strict", "?"):.1%}')
else:
    STARTING_MERGED = 'Qwen/Qwen2.5-1.5B-Instruct'
    B1_REPO = STARTING_MERGED
    PRIOR_STAGE = 'fresh-instruct (no KTO warmup found)'
    print(f'⚠️  no W4 KTO gate result found under {DRIVE_BASE}/kto_runs/')
    print(f'   using raw {STARTING_MERGED} — expect format gate to be the bottleneck')
    print(f'   to enable KTO warmup: run colab/30_train_responder_kto.ipynb first')

print(f'\nstarting from {PRIOR_STAGE}: {STARTING_MERGED}')
print(f'B1 baseline for reward delta: {B1_REPO}')


In [ ]:
# 5) Install deps + pytest pre-flight (incl. new W7 tests).
# Note: flash-attn dropped — building from source on Colab stalled imports for ~30 min.
# SDPA path is ~30% slower but works out-of-box; cell 10 sets attn_implementation='sdpa'.
# bm25s required by mcrs.retrieval_modules.bm25 (cell 8). Same Colab trap as W4 KTO.
!pip install -q --upgrade transformers datasets 'pandas<3.0' tqdm omegaconf
!pip install -q --upgrade 'trl>=0.12.0' 'peft>=0.13.0' 'torchao>=0.16.0' trackio accelerate bm25s
!python -c 'import torch, transformers, trl, peft, bm25s; print("torch", torch.__version__, "trl", trl.__version__, "peft", peft.__version__, "bm25s ok")'

!cd /content/recsys2026 && python -m pytest \
    tests/test_reward_fns.py \
    tests/test_build_grpo_dataset.py \
    tests/test_train_distilled_judge.py \
    tests/test_build_train_plus_dev.py \
    -q

In [ ]:
# 6) Train the distilled judge (~30 min on A100, AUTO-SKIPS if existing repo on Hub).
#
# This is the highest-leverage step in the Option B refactor: replaces
# r_judge_stub (always returns 0) with a cross-encoder that approximates
# Gemini's judgment. Without this, R_judge contributes 0 gradient.
#
# SMOOTH RE-RUN: if today's judge already exists on Hub from a prior run,
# this cell skips the ~30 min retraining and just sets JUDGE_HUB_REPO.
# Override JUDGE_HUB_REPO below if you want to reuse a SPECIFIC judge
# (e.g., yesterday's) instead of today's date.
from datetime import date
from pathlib import Path
from huggingface_hub import repo_exists

JUDGE_HUB_REPO = f'{HF_USERNAME}/recsys2026-distilled-judge-{date.today().isoformat()}'
# To reuse a specific prior judge, uncomment and edit:
# JUDGE_HUB_REPO = f'{HF_USERNAME}/recsys2026-distilled-judge-2026-05-08'
JUDGE_OUT_DIR = '/content/recsys2026/distilled_judge_run'
JUDGE_DATA_DIR = '/content/recsys2026/data/distilled_judge'

# Hub-skip check: if the judge already exists on Hub, don't retrain.
SKIP_JUDGE_TRAIN = False
try:
    if repo_exists(JUDGE_HUB_REPO, repo_type='model', token=HF_TOKEN):
        SKIP_JUDGE_TRAIN = True
        print(f'✓ judge already on Hub — skipping retrain: https://huggingface.co/{JUDGE_HUB_REPO}')
except Exception as e:
    print(f'(repo_exists check failed: {e!r}; will train fresh)')

if not SKIP_JUDGE_TRAIN:
    # data/ is gitignored, so the 297MB anchor parquet is not in the cloned repo.
    # Fetch it from a private HF dataset (one-time upload — see notebook README).
    ANCHORS_REPO = f'{HF_USERNAME}/recsys2026-anchors'
    ANCHORS_LOCAL = '/content/recsys2026/data/reward_calibration_anchors.parquet'
    if not Path(ANCHORS_LOCAL).exists():
        from huggingface_hub import hf_hub_download
        print(f'fetching anchor parquet from {ANCHORS_REPO}…')
        fetched = hf_hub_download(
            repo_id=ANCHORS_REPO, repo_type='dataset',
            filename='reward_calibration_anchors.parquet',
            token=HF_TOKEN, local_dir='/content/recsys2026/data',
        )
        print(f'✓ anchors at {fetched}')
    else:
        print(f'reusing local anchors at {ANCHORS_LOCAL}')

    # Step 6a — prepare JSONL pairs (CPU; ~1 min).
    !cd /content/recsys2026 && python scripts/train_distilled_judge.py \
        --mode prepare \
        --anchors data/reward_calibration_anchors.parquet \
        --out-dir {JUDGE_DATA_DIR} \
        --val-frac 0.1 --seed 42

    # Bang commands don't propagate exit codes — verify prepare actually produced JSONL output.
    import os
    prepare_files = os.listdir(JUDGE_DATA_DIR) if os.path.isdir(JUDGE_DATA_DIR) else []
    if not any(f.endswith('.jsonl') for f in prepare_files):
        raise SystemExit(
            f'❌ prepare step did not produce JSONL files in {JUDGE_DATA_DIR}.\n'
            f'   Scroll up — the !python output above has the real error.'
        )

    # Step 6b — train cross-encoder (~30 min A100).
    !cd /content/recsys2026 && python scripts/train_distilled_judge.py \
        --mode train \
        --data-dir {JUDGE_DATA_DIR} \
        --base-model cross-encoder/ms-marco-MiniLM-L-6-v2 \
        --output-dir {JUDGE_OUT_DIR} \
        --hub-repo {JUDGE_HUB_REPO} \
        --epochs 2 --batch-size 64 --lr 2e-5 --max-length 512

    # Verify a model checkpoint was actually written.
    train_files = os.listdir(JUDGE_OUT_DIR) if os.path.isdir(JUDGE_OUT_DIR) else []
    has_model = any(f.endswith('.safetensors') or f == 'pytorch_model.bin' for f in train_files)
    if not has_model:
        raise SystemExit(
            f'❌ train step did not produce a model in {JUDGE_OUT_DIR}.\n'
            f'   Scroll up — the !python output above has the real error.'
        )
    print(f'✓ distilled judge trained: https://huggingface.co/{JUDGE_HUB_REPO}')


In [ ]:
# 7) Build the pilot GRPO parquet (small scale, AUTO-SKIPS via Drive symlink).
#
# SMOOTH RE-RUN: each sub-step (build_reward_dataset, augment_envelope,
# retrieval pre-compute, build_grpo_dataset) has an `if not Path(...).exists()`
# skip check. The data/ symlink in cell `# 2b)` redirects all writes to Drive,
# so once cell 7 runs successfully ONCE, future runs skip the heavy ~30 min
# retrieval loop entirely. Final parquet stats line still prints either way.
#
# 1500 sessions instead of 15000 → ~10% data. Runs the same pipeline:
# build_reward_dataset → augment_envelope → retrieval pre-compute →
# build_grpo_dataset, but with a smaller --n-sessions.
#
# Reuses cells from notebook 32 cell 6/7 — see that for the full retrieval
# pre-compute logic. We only retrieve for the 1500-session subset to keep
# the pre-compute step under 10 min.
import json, sys, re
from pathlib import Path
import pandas as pd
from tqdm import tqdm

N_SESSIONS_PILOT = 1500
REWARD = '/content/recsys2026/data/reward_train_pilot.parquet'
ENV_PATH = '/content/recsys2026/data/reward_train_envelope_pilot.parquet'
RETRIEVAL_OUT = '/content/recsys2026/data/trl/grpo_retrieval_pilot_v2.parquet'  # v2: adds gold_track_name column
GRPO_OUT = '/content/recsys2026/data/trl/grpo_pilot_v2.parquet'  # v2: adds gold_track_name column

if not Path(REWARD).exists():
    !cd /content/recsys2026 && python scripts/build_reward_dataset.py \
        --hf-split train --n-sessions {N_SESSIONS_PILOT} \
        --split-val 0.1 --out {REWARD}
if not Path(ENV_PATH).exists():
    !cd /content/recsys2026 && python scripts/augment_envelope.py --in {REWARD} --out {ENV_PATH}

# Retrieval pre-compute — same code as notebook 32 cell 7 but smaller input.
if Path(RETRIEVAL_OUT).exists():
    print(f'reusing existing {RETRIEVAL_OUT}')
else:
    sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
    import torch
    from mcrs.retrieval_modules import load_retrieval_module
    from mcrs.rerankers.pro_rank import ProRankReranker
    from mcrs.query_rewriters.cmqr import CMQR_REWRITER
    from mcrs.query_rewriters.state_tracker import StateTracker
    from mcrs.lm_modules import load_lm_module
    from mcrs.db_item import MusicCatalogDB

    ITEM_DB = 'talkpl-ai/TalkPlayData-Challenge-Track-Metadata'
    DATASET = 'talkpl-ai/TalkPlayData-Challenge-Dataset'
    SPLITS = ['all_tracks']
    CORPUS = ['track_name', 'artist_name', 'album_name']
    CACHE = '/content/recsys2026/music-crs-baselines/experiments/cache'
    LM_TYPE = 'meta-llama/Llama-3.2-1B-Instruct'
    PROMPTS = '/content/recsys2026/music-crs-baselines/mcrs/system_prompts'

    lm = load_lm_module(lm_type=LM_TYPE, device='cuda', attn_implementation='sdpa', dtype=torch.bfloat16, use_vllm=False)
    retrieval = load_retrieval_module('wrrf_bm25_dense_lyrics_v1', ITEM_DB, SPLITS, CORPUS, CACHE)
    state_tracker = StateTracker(lm=lm, prompt_path=f'{PROMPTS}/state_extraction.txt', cache_dir=CACHE, max_new_tokens=96)
    cmqr = CMQR_REWRITER(lm=lm, inner_retriever=retrieval, prompt_path=f'{PROMPTS}/cmqr_rewrites.txt',
                         cache_dir=CACHE, n_rewrites=4, topk_per_rewrite=50, rrf_k=60, max_new_tokens=96)
    reranker = ProRankReranker(item_db_name=ITEM_DB, track_split_types=SPLITS, corpus_types=CORPUS, cache_dir=CACHE, with_rationales=True)
    item_db = MusicCatalogDB(dataset_name=ITEM_DB, split_types=SPLITS, corpus_types=CORPUS)
    valid_catalog = set(item_db.metadata_dict.keys())

    env_df = pd.read_parquet(ENV_PATH).query('label == 1').drop_duplicates(['session_id', 'turn_number'])
    print(f'unique POS turns: {len(env_df):,}')

    from datasets import load_dataset
    raw = load_dataset(DATASET, split='train')
    sid2turn_to_gold = {}
    sid2turn_to_user_query = {}
    sid2turn_to_history = {}
    for sess in raw:
        sid = sess['session_id']
        msgs = sorted(sess['conversations'], key=lambda m: (int(m['turn_number']), m['role']))
        for msg in msgs:
            if msg['role'] == 'music':
                sid2turn_to_gold[(sid, int(msg['turn_number']))] = str(msg['content'])
        for tn in range(1, 9):
            prior = [m for m in msgs if int(m['turn_number']) < tn]
            history = '\n'.join(f"{m['role']}: {m['content']}" for m in prior)
            user_msg = next((m for m in msgs if int(m['turn_number']) == tn and m['role'] == 'user'), None)
            if user_msg is None: continue
            sid2turn_to_user_query[(sid, tn)] = str(user_msg['content'])
            sid2turn_to_history[(sid, tn)] = history

    rows_out = []
    BATCH = 16
    rows_buffer = []
    for i, row in enumerate(tqdm(env_df.itertuples(index=False), total=len(env_df), desc='retrieve')):
        rows_buffer.append((row.session_id, int(row.turn_number)))
        if len(rows_buffer) < BATCH and i + 1 < len(env_df): continue
        sids = [s for s, _ in rows_buffer]; tns = [t for _, t in rows_buffer]
        queries = [sid2turn_to_user_query.get((s, t), '') for s, t in rows_buffer]
        histories = [sid2turn_to_history.get((s, t), '') for s, t in rows_buffer]
        # Batched state extraction (one LM call, not 16) — falls back to per-row on backend mismatch.
        try:
            states = state_tracker.batch_extract(sids, tns, queries, histories)
        except Exception:
            states = []
            for s, t, q, h in zip(sids, tns, queries, histories):
                try: states.append(state_tracker.extract(s, t, q, h))
                except Exception: states.append(None)
        cmqr.set_batch_context(session_ids=sids, turn_numbers=tns, extracted_states=states)
        try: top100 = cmqr.batch_text_to_item_retrieval(queries, topk=100, user_ids=[None]*len(queries))
        except TypeError: top100 = cmqr.batch_text_to_item_retrieval(queries, topk=100)
        top20 = reranker.rerank(queries, top100, topk=20)
        # Pass 1: catalog-filter + dedupe per row → kept_per_row.
        kept_per_row = []
        for s, t, q, ids20, pool in zip(sids, tns, queries, top20, top100):
            seen, kept = set(), []
            for tid in ids20:
                if tid in seen or tid not in valid_catalog: continue
                kept.append(tid); seen.add(tid)
            if len(kept) < 20:
                for tid in pool:
                    if len(kept) >= 20: break
                    if tid in seen or tid not in valid_catalog: continue
                    kept.append(tid); seen.add(tid)
            kept_per_row.append(kept[:20])
        # Batched rationale generation — one flattened generate over 16 × 20 = 320 prompts
        # at bs=64, replacing 16 separate generate_rationales() calls. Falls back to per-row.
        try:
            rats_per_row = reranker.batch_generate_rationales(queries, kept_per_row, batch_size=64)
        except Exception:
            rats_per_row = []
            for q, kept in zip(queries, kept_per_row):
                try: rats_per_row.append(reranker.generate_rationales(q, kept))
                except Exception: rats_per_row.append([''] * len(kept))
        # Pass 2: emit row dicts using the precomputed kept + rationales.
        for s, t, q, kept, rats in zip(sids, tns, queries, kept_per_row, rats_per_row):
            top1_tid = kept[0] if kept else ''
            top1_meta = item_db.metadata_dict.get(top1_tid, {}) if top1_tid else {}
            tn_get = lambda f: (top1_meta.get(f) or [''])
            tn1 = (tn_get('track_name')[0] if isinstance(tn_get('track_name'), list) else str(tn_get('track_name'))) or ''
            an1 = (tn_get('artist_name')[0] if isinstance(tn_get('artist_name'), list) else str(tn_get('artist_name'))) or ''
            # Algo-review fix (2026-05-11): look up gold track name from item_db so
            # the reward closure can compute per-completion grounding (r_retr_grounded).
            # Falls back to '' when gold isn't in the catalog (rare).
            gold_id = sid2turn_to_gold.get((s, t), '')
            gold_meta = item_db.metadata_dict.get(gold_id, {}) if gold_id else {}
            gn_field = gold_meta.get('track_name')
            gold_name = (gn_field[0] if isinstance(gn_field, list) and gn_field else str(gn_field or '')) or ''
            rows_out.append({
                'session_id': s, 'turn_number': t,
                'gold_track_id': gold_id, 'gold_track_name': gold_name,
                'predicted_track_ids': kept, 'top1_track_name': tn1,
                'top1_artist_name': an1, 'reranker_rationales': rats,
            })
        rows_buffer.clear()

    out_df = pd.DataFrame(rows_out).drop_duplicates(['session_id', 'turn_number'], keep='last')
    Path(RETRIEVAL_OUT).parent.mkdir(parents=True, exist_ok=True)
    out_df.to_parquet(RETRIEVAL_OUT, index=False)
    print(f'✓ retrieval cache: {len(out_df):,} rows')

# Build the GRPO parquet via the conversational-prompt path.
SYSTEM_PROMPT_TXT = '/content/recsys2026/data/trl/grpo_system_prompt.txt'
PROMPTS_DIR = '/content/recsys2026/music-crs-baselines/mcrs/system_prompts'
with open(f'{PROMPTS_DIR}/roleplay.txt', encoding='utf-8') as f: rp = f.read()
with open(f'{PROMPTS_DIR}/response_generation_cot_user_state.txt', encoding='utf-8') as f: cp = f.read()
Path(SYSTEM_PROMPT_TXT).parent.mkdir(parents=True, exist_ok=True)
with open(SYSTEM_PROMPT_TXT, 'w', encoding='utf-8') as f: f.write(rp + '\n\n' + cp)

if not Path(GRPO_OUT).exists():
    !cd /content/recsys2026 && python scripts/build_grpo_dataset.py \
        --envelope {ENV_PATH} --retrieval {RETRIEVAL_OUT} \
        --system-prompt-path {SYSTEM_PROMPT_TXT} --out {GRPO_OUT}

import pandas as pd
d = pd.read_parquet(GRPO_OUT)
print(f'\npilot GRPO: {len(d):,} rows')

In [ ]:
# 8) Schema + 90/10 split.
from datasets import Dataset
import numpy as np

def _norm(p):
    if isinstance(p, np.ndarray): return [dict(m) for m in p]
    if isinstance(p, list) and p and isinstance(p[0], np.ndarray): return [dict(m) for m in p]
    return p
d['prompt'] = d['prompt'].apply(_norm)
ds = Dataset.from_pandas(d, preserve_index=False)
split = ds.train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = split['train'], split['test']
print(f'train: {len(train_ds):,}  eval: {len(eval_ds):,}')

In [ ]:
# 9) Trackio init + GRPO pilot training (Option B reward closure).
#
# This is where the Option B refactor lands at training:
#   - reward_main calls compose_r_turn(judge_score=judge.score(ctx, response),
#     group_responses=...) with the new weights.
#   - judge is a DistilledJudge instance loading the cross-encoder we trained
#     in cell 6.
#   - reward_format is a separate fn (kept from W6 review P1-2).
#
# 2026-05-11: switched to Qwen-2.5-1.5B-Instruct (fresh, no W4/W5 chain).
# 1.5B's smaller footprint lets us keep G=4 comfortably on A100-40GB. Effective trajectories per opt step unchanged (4 micro-
# batches × G=4 = 16). gradient_checkpointing stays True — generation +
# KV cache for G=4 still pushes memory, recipe says don't disable for GRPO.
# Free GPU memory from cell 7 — retrieval pre-compute leaves ~30 GB of
# Llama-1B + dense retrievers + ProRank weights on GPU that GRPO doesn't need.
import gc, torch
for v in ['lm', 'state_tracker', 'cmqr', 'reranker', 'item_db', 'retrieval', 'raw',
          'sid2turn_to_gold', 'sid2turn_to_user_query', 'sid2turn_to_history']:
    if v in globals():
        try: del globals()[v]
        except Exception: pass
gc.collect()
torch.cuda.empty_cache()
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f'GPU free before GRPO load: {free/1e9:.1f} / {total/1e9:.1f} GB')

from datetime import date
import trackio
from transformers import TrainerCallback

RUN_NAME = f'b3-grpo-pilot-{date.today().isoformat()}-qwen3b-v5-kto'  # v5: back to Qwen-3B (1.5B hit format ceiling); KTO-warmed + grounded reward
TRACKIO_OK = True
try:
    trackio.init(project='recsys2026', name=RUN_NAME, group='b-stage',
                 config={'pilot': True, 'judge': JUDGE_HUB_REPO, 'starting': STARTING_MERGED,
                         'n_sessions': N_SESSIONS_PILOT, 'max_steps': 400,
                         'per_device_train_batch_size': 1, 'gradient_accumulation_steps': 4,
                         'beta': 0.10, 'lr_scheduler': 'constant_with_warmup', 'grounded_retr': True})
    print(f'✓ Trackio: {RUN_NAME}')
except Exception as e:
    TRACKIO_OK = False
    print(f'⚠️  trackio failed: {e!r}')

class TrackioCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not TRACKIO_OK or not logs: return
        try: trackio.log({k: float(v) for k, v in logs.items() if isinstance(v, (int, float))})
        except Exception: pass

import torch, gc, json, sys
from peft import LoraConfig
from trl import GRPOTrainer, GRPOConfig
sys.path.insert(0, '/content/recsys2026/scripts')
from reward_fns import compose_r_turn, r_format, DistilledJudge

# Instantiate the distilled judge — lazy-loads on first .score() call.
JUDGE = DistilledJudge(checkpoint=JUDGE_HUB_REPO)
# Deep-review P1-4: warmup before trainer init.
JUDGE.warmup()

HUB_REPO = f'{HF_USERNAME}/recsys2026-{RUN_NAME}'
OUTPUT_DIR = f'/content/recsys2026/training_runs/{RUN_NAME}'

peft_config = LoraConfig(r=32, lora_alpha=32, lora_dropout=0.05,
                         bias='none', task_type='CAUSAL_LM', target_modules='all-linear')

def _completion_text(c):
    if isinstance(c, list) and c:
        last = c[-1]
        if isinstance(last, dict): return str(last.get('content', ''))
        return str(last)
    return str(c)


def _user_content_from_prompt(prompt):
    """Extract the user-role content string for the judge's `context` arg."""
    if isinstance(prompt, list):
        for msg in prompt:
            if isinstance(msg, dict) and msg.get('role') == 'user':
                return str(msg.get('content', ''))
        return ''
    return str(prompt)


def reward_main(prompts, completions, **kwargs):
    """Composite Option B reward (no format gate, format split below)."""
    # Group completion indices by (session_id, turn_number) for the
    # group_responses (intra-rollout diversity) bonus.
    from collections import defaultdict
    groups = defaultdict(list)
    sids = kwargs['session_id']; tns = kwargs['turn_number']
    for i in range(len(completions)):
        groups[(sids[i], tns[i])].append(i)

    # Pre-compute user-role contexts for the judge in one batch.
    contexts = [_user_content_from_prompt(p) for p in prompts]
    completion_texts = [_completion_text(c) for c in completions]
    # Batch the judge calls — the cross-encoder forward pass is the bulk
    # of the per-step cost. score_batch returns 0.0 per row if no checkpoint.
    judge_scores = JUDGE.score_batch(contexts, completion_texts, batch_size=16)

    scores = []
    for i, completion in enumerate(completions):
        # Peer responses for diversity bonus.
        peer_indices = groups[(sids[i], tns[i])]
        peer_responses = [completion_texts[j] for j in peer_indices]
        comps = compose_r_turn(
            predicted_track_ids=list(kwargs['predicted_track_ids'][i]),
            gold_track_id=kwargs['gold_track_id'][i],
            response_text=completion_texts[i],
            valid_catalog=None,
            top1_meta=json.loads(kwargs['top1_meta_json'][i]),
            user_state=json.loads(kwargs['user_state_json'][i]),
            user_profile=json.loads(kwargs['user_profile_json'][i]) if 'user_profile_json' in kwargs else None,  # gap-analysis Step 3: user_profile piped
            history_text=kwargs['history_text'][i],
            judge_score=judge_scores[i],
            include_format=False,
            group_responses=peer_responses,
            # Algo-review fix (2026-05-11): per-completion grounding signal so
            # W_RETR (largest weight) contributes to within-GRPO-group advantage.
            gold_track_name=kwargs.get('gold_track_name', [''] * len(completions))[i] if 'gold_track_name' in kwargs else None,
        )
        scores.append(comps['r_turn'])
    return scores


def reward_format(prompts, completions, **kwargs):
    return [r_format(_completion_text(c)) for c in completions]


# Fast-iteration GRPO config (G=4 per TRL default, no vLLM).
# Tunable knobs (v2-grounded — 2026-05-11):
#   max_steps=400       → covers the useful regime; v2's reward peak was ~step 100. ~15-25 min on A100 with 1.5B base.
#   beta=0.10           → tighter KL anchor (was 0.04); Rec-R1 §4.2 default for judge-dominated rewards.
#   lr_scheduler=constant_with_warmup → was cosine; avoids decaying LR through the broken late regime.
#   num_generations=4   → TRL default. Per-completion grounding now gives intra-group variance (was the root cause of v2 mode collapse).
#   max_completion_length=256 → was 128 in v1; eliminates mid-sentence truncation observed in eval samples.
#   per_device × grad_accum × world_size MUST be divisible by num_generations (TRL constraint).
#
# vLLM dropped from this config — Colab's vllm-in-Jupyter is fragile (fileno UnsupportedOperation,
# version mismatch with TRL). Stays as a future option if you install a pinned vllm 0.12-0.18 outside
# the notebook and add use_vllm=True back.
config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    model_init_kwargs={"torch_dtype": "bfloat16", "attn_implementation": "sdpa", "device_map": "cuda"},
    push_to_hub=True, hub_model_id=HUB_REPO, hub_strategy='every_save', hub_private_repo=True,
    num_generations=4, scale_rewards=False, max_completion_length=256,  # was 128 — eliminate truncation observed in v1 pilot samples
    temperature=0.9, beta=0.10, reward_weights=[0.90, 0.10],  # v4: format weight back to 0.10 (KTO base should make format compliant; reclaim weight for grounding/judge)
    max_steps=400,                           # was 1000 — v2 peak reward was at training step ~100; 400 covers the useful regime
    per_device_train_batch_size=1, gradient_accumulation_steps=4,  # 1 × 4 × 1 % 4 == 0 ✓
    learning_rate=5e-6, lr_scheduler_type='constant_with_warmup', warmup_ratio=0.10,  # was cosine — constant LR avoids spending muscle on the broken regime
    bf16=True, gradient_checkpointing=True,
    eval_strategy='no',
    save_strategy='steps', save_steps=100, save_total_limit=2,
    logging_steps=10, report_to='none',
    seed=42, data_seed=42,
)

callbacks = [TrackioCallback()] if TRACKIO_OK else []
trainer = GRPOTrainer(
    model=STARTING_MERGED, args=config,
    train_dataset=train_ds, eval_dataset=eval_ds,
    reward_funcs=[reward_main, reward_format],
    peft_config=peft_config, callbacks=callbacks,
)
print(f'\U0001f680 PILOT GRPO (~15-25 min A100 expected on Qwen-1.5B with max_steps=400)…')

# Auto-resume if a checkpoint already exists in OUTPUT_DIR (idempotent re-run).
from pathlib import Path as _Path
_has_ckpt = any(_Path(OUTPUT_DIR).glob('checkpoint-*'))
if _has_ckpt:
    print(f'   ↻ found existing checkpoint(s) under {OUTPUT_DIR} — resuming')
trainer.train(resume_from_checkpoint=_has_ckpt)
print('✓ pilot training complete')

In [ ]:
# 10) Push adapter + in-place merge.
import gc, torch
from transformers import AutoTokenizer

trainer.push_to_hub()
trainer.optimizer = None; trainer.lr_scheduler = None
gc.collect(); torch.cuda.empty_cache()

fully_merged = trainer.model.merge_and_unload()
tok = AutoTokenizer.from_pretrained(STARTING_MERGED)
MERGED_REPO = f'{HF_USERNAME}/recsys2026-{RUN_NAME}-merged'
fully_merged.push_to_hub(MERGED_REPO, private=True,
                         commit_message=f'PILOT W6 Option B merged on {STARTING_MERGED}')
tok.push_to_hub(MERGED_REPO, private=True)
print(f'✓ pilot merged: https://huggingface.co/{MERGED_REPO}')

In [ ]:
# 11) Format compliance + reward delta vs B1 — RELAXED PILOT GATE.
#
# Pilot gate: format ≥ 90% (not 95%); Δ R_turn ≥ +0.015 (not +0.03).
import json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import sys
sys.path.insert(0, '/content/recsys2026/scripts')
from reward_fns import r_format, ENVELOPE, compose_r_turn

w6_tok = AutoTokenizer.from_pretrained(MERGED_REPO)
w6_model = AutoModelForCausalLM.from_pretrained(MERGED_REPO, torch_dtype=torch.bfloat16, device_map='cuda').eval()
b1_model = None
if B1_REPO:
    b1_model = AutoModelForCausalLM.from_pretrained(B1_REPO, torch_dtype=torch.bfloat16, device_map='cuda').eval()

def gen(model, tok, conv):
    formatted = tok.apply_chat_template(conv, tokenize=False, add_generation_prompt=True)
    enc = tok(formatted, return_tensors='pt', truncation=True, max_length=2048).to('cuda')
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=320, do_sample=False,
                             pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0, enc['input_ids'].shape[1]:], skip_special_tokens=True)

n_eval = min(50, len(eval_ds))
n_strict = n_loose = 0
samples = []
w6_rs, b1_rs = [], []
for i in range(n_eval):
    ex = eval_ds[i]
    out = gen(w6_model, w6_tok, ex['prompt'])
    if r_format(out) == 1.0: n_strict += 1
    if ENVELOPE.search(out): n_loose += 1
    if len(samples) < 3: samples.append(out[:400])
    w6_rs.append(compose_r_turn(
        predicted_track_ids=list(ex['predicted_track_ids']),
        gold_track_id=ex['gold_track_id'], response_text=out, valid_catalog=None,
        top1_meta=json.loads(ex['top1_meta_json']),
        user_state=json.loads(ex['user_state_json']),
        history_text=ex['history_text'], judge_score=JUDGE.score(
            ex['prompt'][1]['content'] if isinstance(ex['prompt'], list) and len(ex['prompt']) > 1 else '', out
        ),
    )['r_turn'])
    if b1_model:
        b1_out = gen(b1_model, w6_tok, ex['prompt'])
        b1_rs.append(compose_r_turn(
            predicted_track_ids=list(ex['predicted_track_ids']),
            gold_track_id=ex['gold_track_id'], response_text=b1_out, valid_catalog=None,
            top1_meta=json.loads(ex['top1_meta_json']),
            user_state=json.loads(ex['user_state_json']),
            history_text=ex['history_text'], judge_score=JUDGE.score(
                ex['prompt'][1]['content'] if isinstance(ex['prompt'], list) and len(ex['prompt']) > 1 else '', b1_out
            ),
        )['r_turn'])

format_strict = n_strict / n_eval
format_loose = n_loose / n_eval
mean_w6 = sum(w6_rs) / max(len(w6_rs), 1)
mean_b1 = sum(b1_rs) / max(len(b1_rs), 1) if b1_rs else None
delta = (mean_w6 - mean_b1) if mean_b1 is not None else None

print(f'\nPILOT format strict: {format_strict:.1%} (gate: ≥90%)')
print(f'PILOT format loose:   {format_loose:.1%}')
print(f'PILOT W6 R_turn:      {mean_w6:.4f}')
if mean_b1 is not None:
    print(f'PILOT B1 R_turn:      {mean_b1:.4f}')
    print(f'PILOT Δ vs B1:        {delta:+.4f}  (gate: ≥+0.015)')

print('\nSamples:')
for i, s in enumerate(samples, 1): print(f'\n--- {i} ---\n{s}')

gate_format = format_strict >= 0.90
gate_reward = (delta is not None and delta >= 0.015)

print('\n' + '=' * 60)
if gate_format and gate_reward:
    print(f'PILOT PASS — schedule full W6 (notebook 32, 12k steps).')
elif gate_format:
    print(f'PILOT PARTIAL — format OK, reward delta below threshold. Tune & re-pilot.')
else:
    print(f'PILOT FAIL — format regressed. Revisit reward weights or fall back to W4.')
print('=' * 60)

# Persist gate result.
import os
import numpy as np
def _clean(o):
    if isinstance(o, dict): return {k: _clean(v) for k, v in o.items()}
    if isinstance(o, list): return [_clean(x) for x in o]
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return o
result = {
    'stage': 'B3-pilot',
    'run_name': RUN_NAME,
    'starting_base': STARTING_MERGED,
    'judge': JUDGE_HUB_REPO,
    'merged_hub_model': MERGED_REPO,
    'format_compliance_strict': format_strict,
    'mean_w6_r_turn': mean_w6,
    'mean_b1_r_turn': mean_b1,
    'delta_r_turn_vs_b1': delta,
    'gate_format_passed': gate_format,
    'gate_reward_passed': gate_reward,
    'pilot_passed': gate_format and gate_reward,
    'sample_outputs': samples,
}
out_path = f'{DRIVE_BASE}/grpo_pilot_runs/{RUN_NAME}/gate_result.json'
os.makedirs(os.path.dirname(out_path), exist_ok=True)
with open(out_path, 'w', encoding='utf-8') as f: json.dump(_clean(result), f, ensure_ascii=False, indent=2)
print(f'\ngate_result → {out_path}')
if globals().get('TRACKIO_OK'):  # guard for runtime-restart cases (cell 9 may not have run)
    try:
        trackio.finish()
    except Exception:
        pass

## Pilot decision tree

**PILOT PASS** (format ≥ 90% AND Δ R_turn ≥ +0.015):
- Schedule full W6 via `colab/32_train_responder_grpo.ipynb` (12k steps, 10 A100-hr).
- The full run can reuse this notebook's distilled judge — set `JUDGE_HUB_REPO` in cell 11 of notebook 32.
- Optional: also run `colab/41_run_blindset_A.ipynb` against the pilot merged model for cheap Gemini-judge calibration.

**PILOT PARTIAL** (format OK, reward delta short):
- Likely cause: 2k steps wasn't enough convergence. Try doubling to 4k and re-piloting. ~3 A100-hr.
- Or: nudge `learning_rate` 5e-6 → 1e-5; raise `kl_beta` if format started slipping near the end.

**PILOT FAIL** (format regressed):
- Revert: ship W4-merged or W5-merged directly to Blind-A as a no-RL baseline; skip W6 entirely.
- Likely cause: reward weights too aggressive. Test A/B by reverting to v1 weights (R_retr=0.70) for one run, OR halve `W_JUDGE` (the new piece) to see if the judge is mis-calibrated.